#      RAG SYSTEM

### document loading

In [1]:

import pathlib

def load_documents(directory_path):
    docs=[]
    path = pathlib.Path(directory_path)
    
    if not path.exists() or not path.is_dir():
        raise ValueError(f"Invalid directory path: {directory_path}")
    
    for file_path in path.iterdir():
        if file_path.suffix in ['.txt', '.md']:
            try:
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                    if not content.strip():
                        print(f"Warning: {file_path} is empty.")
                    metadata = {
                        'file_name': file_path.name,
                        "source": str(file_path),
                    }
                    docs.append({"content": content, "metadata": metadata})
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return docs

loaded_docs = load_documents(r"C:\AI_PROJECTS\ice_pytorch\data")
loaded_docs[:2]


[{'content': '# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI\'s `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, but geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in high-dimensional spaces. \n\nAs the number of dimensions increases, the volume of the space increases so rapidly that the available data becomes sparse. More importantly for retrieval systems, the concept of "distance" becomes less meanin

### chunking

In [2]:
import re
import bisect

def get_sections_with_hierarchy(content):
    # 1. Clean the content of windows style carriage returns just in case
    cleaned_content = content.replace('\r\n', '\n')
    
    # 2. Resilient pattern: Looks for 1 to 6 '#' signs at the start of any line
    # followed by at least one space or tab, capturing the text until the end of that line.
    header_pattern = re.compile(r'^(#{1,6})[ \t]+(.*)$', re.MULTILINE)
    
    sections = []
    hierarchy_stack = []

    for match in header_pattern.finditer(cleaned_content):
        # match.group(1) is the actual hashtags string (e.g. "##")
        level = len(match.group(1))        
        header_text = match.group(2).strip() 
        start_char = match.start()
        
        # Maintain the stack hierarchy
        while hierarchy_stack and hierarchy_stack[-1][0] >= level:
            hierarchy_stack.pop()
            
        hierarchy_stack.append((level, header_text))
        hierarchy_path = " > ".join([h[1] for h in hierarchy_stack])
        
        sections.append({
            "start_char": start_char,
            "section_path": hierarchy_path
        })
        
    return sections
'''
header_data=[]
for i in loaded_docs:
    header_data.append(get_sections_with_hierarchy(i["content"]))
    '''

def fixed_size_chunking(documents,n=0, chunk_size=512):
    chunks=[]
    content=documents[n]["content"]
    file_name=documents[n]["metadata"]["file_name"]
    header_info=get_sections_with_hierarchy(content)
    header_starts=[h["start_char"] for h in header_info]
    for i in range(0, len(content), chunk_size):
        chunk=content[i:i+chunk_size]

        if chunk:
            chunks.append({"content": chunk,
                          "metadata": {
                                "file_name": file_name,
                                "chunking_strategy": "fixed",
                                "chunk_index": i//chunk_size,
                                "start_char": i,
                                "section": header_info[bisect.bisect_right(header_starts, i) - 1]["section_path"] if header_info and bisect.bisect_right(header_starts, i) > 0 else "",
                                "words": len(chunk.split()),
                                "lines": len(chunk.splitlines())
                          }}
                          )
    return chunks

def sliding_window_chunking(documents,n=0, chunk_size=512, overlap=128):
    chunks=[]
    file_name=documents[n]["metadata"]["file_name"]
    step=chunk_size-overlap
    header_info=get_sections_with_hierarchy(documents[n]["content"])
    header_starts=[h["start_char"] for h in header_info]
    for i in range(0, len(documents[n]["content"]), step):
        chunk=documents[n]["content"][i:i+chunk_size]
        if chunk:
            chunks.append({"content": chunk,
                          "metadata": {
                                "file_name": file_name,
                                "chunking_strategy": "sliding",
                                "chunk_index": i//chunk_size,
                                "section": header_info[bisect.bisect_right(header_starts, i) - 1]["section_path"] if header_info and bisect.bisect_right(header_starts, i) > 0 else "",
                                "start_char": i,
                                "end_char": i+len(chunk),
                                "characters": len(chunk),
                                "words": len(chunk.split()),
                                "lines": len(chunk.splitlines())
                          }}
                          )
    return chunks

def header_aware_chunking(documents):
    chunks=[]
    for i, doc in enumerate(documents):
        sections = re.split(r'\n(?=#)', doc["content"])
        file_name=doc["metadata"]["file_name"]
        header_info=get_sections_with_hierarchy(doc["content"])
        header_starts=[h["start_char"] for h in header_info]
        current_char_pos = 0
        for idx, section in enumerate(sections):
            if section.strip():
                section_start = doc["content"].find(section, current_char_pos)
                pos = bisect.bisect_right(header_starts, section_start)
                section_path = (
                    header_info[pos - 1]["section_path"]
                    if header_info and pos > 0
                    else ""
                )
                clean_content = re.sub(r'^\s*#+.*(?:\n|$)', '', section).strip()
                chunks.append({
                    "content": clean_content,
                    "metadata": {
                        "file_name": file_name,
                        "chunking_strategy": "header_aware",
                        "chunk_index": idx,
                        "section": section_path,
                        "characters": len(clean_content),
                        "words": len(clean_content.split()),
                        "lines": len(clean_content.splitlines())
                    }
                })
                current_char_pos = section_start + len(section)
    return chunks

In [3]:
for i in loaded_docs:
    print(get_sections_with_hierarchy(i["content"]))

[{'start_char': 0, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)'}, {'start_char': 66, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 1. The Geometry of High-Dimensional Space'}, {'start_char': 593, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 1. The Geometry of High-Dimensional Space > 1.1 The Curse of Dimensionality'}, {'start_char': 1372, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics'}, {'start_char': 1495, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.1 Euclidean Distance (L2)'}, {'start_char': 1736, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.2 Cosine Similarity'}, {'start_char': 2196, 'section_path': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN) > 2. Distance Metrics > 2.3 Inner Prod

In [4]:
def preprocess_document(documents, n=0,chunking_strategy='fixed', chunk_size=512, overlap=128):
    if chunking_strategy == 'fixed':
        return fixed_size_chunking(documents, n, chunk_size)
    elif chunking_strategy == 'sliding':
        return sliding_window_chunking(documents, n, chunk_size, overlap)
    elif chunking_strategy == 'header_aware':
        return header_aware_chunking(documents)
    else:
        raise ValueError(f"Unknown chunking strategy: {chunking_strategy}")

In [5]:
def preprocess_documents(documents, chunking_strategy='fixed', chunk_size=512, overlap=128):
    all_chunks=[]
    for n in range(len(documents)):
        
        if(chunking_strategy == "header_aware"):
            chunks=preprocess_document(documents, n, chunking_strategy, chunk_size, overlap)
            all_chunks.extend(chunks)
            break
        
        chunks=preprocess_document(documents, n, chunking_strategy, chunk_size, overlap)
        all_chunks.extend(chunks)
    print(f"Total chunks created using {chunking_strategy}: {len(all_chunks)}")
    return all_chunks

In [6]:
processed_docs = preprocess_documents(loaded_docs, chunking_strategy='fixed', chunk_size=512)
 # Print first 100 characters of each chunk
 

Total chunks created using fixed: 51


In [7]:
print(f"Total documents loaded: {len(loaded_docs)}")
print(f"Total chunks created: {len(processed_docs)}")
for i, chunk in enumerate(processed_docs):
    print(f"\nChunk {i+1}:\n{chunk['content'][:10]}...") 

Total documents loaded: 5
Total chunks created: 51

Chunk 1:
# Advanced...

Chunk 2:
t geometry...

Chunk 3:
between th...

Chunk 4:
n distance...

Chunk 5:
ed with wo...

Chunk 6:
rest Neigh...

Chunk 7:
ains in sp...

Chunk 8:
eaviate.

...

Chunk 9:
ps.

### 4...

Chunk 10:
d entry po...

Chunk 11:
imum on La...

Chunk 12:
dataset to...

Chunk 13:
 the syste...

Chunk 14:
le, a 1024...

Chunk 15:
chine.

--...

Chunk 16:
oat32')

#...

Chunk 17:
5 nearest ...

Chunk 18:
-NN finds ...

Chunk 19:
ply metada...

Chunk 20:
 scan.

##...

Chunk 21:
# Neural N...

Chunk 22:
buted to t...

Chunk 23:
update ste...

Chunk 24:
calability...

Chunk 25:
commonly u...

Chunk 26:
ecution un...

Chunk 27:
nse numeri...

Chunk 28:
ine:

1. Q...

Chunk 29:
imilarity ...

Chunk 30:
quality ev...

Chunk 31:
# Building...

Chunk 32:
retrieval ...

Chunk 33:
becomes im...

Chunk 34:
the previo...

Chunk 35:
uire objec...

Chunk 36:
 but also ...

Chunk 37:
nk boundar...

Chunk 38:
guage mode...

Chunk 3

In [8]:
loaded_docs

[{'content': '# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI\'s `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, but geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in high-dimensional spaces. \n\nAs the number of dimensions increases, the volume of the space increases so rapidly that the available data becomes sparse. More importantly for retrieval systems, the concept of "distance" becomes less meanin

In [9]:
print(type(processed_docs))
print(type(loaded_docs))

<class 'list'>
<class 'list'>


In [10]:
c=0
for i in processed_docs:
    print(f"Chunk {c}: {i} \n")
    c += 1

Chunk 0: {'content': "# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI's `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, bu", 'metadata': {'file_name': 'advanced_vector_search.md', 'chunking_strategy': 'fixed', 'chunk_index': 0, 'start_char': 0, 'section': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)', 'words': 67, 'lines': 7}} 

Chunk 1: {'content': 't geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and 

In [11]:
from fastembed import TextEmbedding
import numpy as np

def get_similarity(v1,v2):
    return np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))



In [13]:
import shutil

shutil.rmtree(
    r"C:\Users\Laksh\AppData\Local\Temp\fastembed_cache",
    ignore_errors=True
)

In [14]:
model=TextEmbedding(model_name="BAAI/bge-base-en")
def model_embedding(processed_docs):
    model=TextEmbedding(model_name="BAAI/bge-base-en")
    embeddings=list(model.embed([chunk['content'] for chunk in processed_docs ]))
    return embeddings

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\AI_PROJECTS\ice_pytorch\monarch\Monarch-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Laksh\AppData\Local\Temp\fastembed_cache\models--Qdrant--fast-bge-base-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\AI_PROJECTS\ice_pytorch\monarch\Monarch-RAG\.venv\Lib\site-packages\fastembed

In [15]:

embeddings=model_embedding(processed_docs)
embeddings

[array([-3.17431092e-02, -7.57644791e-03, -3.55212530e-03,  1.95773095e-02,
         2.70902980e-02, -2.34068986e-02,  3.03278640e-02, -5.92031330e-03,
        -3.53389047e-02, -3.08675300e-02, -4.22625802e-03, -1.73655525e-02,
        -7.65695795e-02,  2.31402367e-02,  5.36494469e-03,  5.80989793e-02,
         2.32525971e-02,  2.30606785e-03,  1.96087584e-02,  4.84950654e-03,
         6.30499516e-03, -2.41830666e-02, -1.53263630e-02,  3.51710059e-02,
         4.82816994e-02, -4.77064075e-03,  1.87246650e-02, -6.42947946e-03,
        -4.48053367e-02, -1.59977598e-03,  3.33912880e-03, -2.66250223e-02,
        -2.25874595e-02, -5.41555956e-02,  3.43340598e-02, -2.42504440e-02,
        -7.99313635e-02,  4.88666911e-03,  1.03828975e-03, -1.38702383e-02,
        -4.09727134e-02, -1.62854102e-02, -3.77577916e-02,  3.22987214e-02,
        -3.46174836e-02,  1.85909569e-02, -1.13431498e-01,  2.20189262e-02,
        -9.25809983e-03,  2.90904148e-03, -8.16801563e-02,  4.65623476e-02,
         5.0

In [16]:
print(f"Semantic matching test (Chunks 0 & 1): {get_similarity(embeddings[0], embeddings[1])}")
print(f"Semantic matching test (Chunks 0 & 2): {get_similarity(embeddings[0], embeddings[2])}")
print(f"Semantic matching test (Chunks 1 & 2): {get_similarity(embeddings[1], embeddings[2])}")

Semantic matching test (Chunks 0 & 1): 0.847450852394104
Semantic matching test (Chunks 0 & 2): 0.8344564437866211
Semantic matching test (Chunks 1 & 2): 0.8378292918205261


In [17]:
sliding_window_chunks = preprocess_documents(loaded_docs, chunking_strategy='sliding', chunk_size=512, overlap=128)
header_aware_chunks = preprocess_documents(loaded_docs, chunking_strategy='header_aware', chunk_size=512, overlap=128)

# header_aware_chunks

Total chunks created using sliding: 69
Total chunks created using header_aware: 82


In [18]:


sliding_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)


In [19]:
def similarity_comparison(query, embeddings):
    query_embedding=list(model.embed([query]))[0]
    for i, emb in enumerate(embeddings):
        sim=get_similarity(query_embedding, emb)
        print(f"Similarity between query and chunk {i}: {sim}")

In [20]:
def similarity_comparison_display_all(query):
    print(f"\nQuery: {query}\n")
    print("Similarity with fixed size chunks:")
    similarity_comparison(query,embeddings)
    print("\nSimilarity with sliding window chunks:")
    similarity_comparison(query,sliding_embeddings)
    print("\nSimilarity with header aware chunks:")
    similarity_comparison(query,header_aware_embeddings)

In [21]:
query1="what is stochastic gradient descent?"
query2="explain the concept of overfitting in machine learning"
query3="what is cosine similarity?"


In [22]:
similarity_comparison_display_all(query1)
similarity_comparison_display_all(query2)
similarity_comparison_display_all(query3)


Query: what is stochastic gradient descent?

Similarity with fixed size chunks:
Similarity between query and chunk 0: 0.7565758228302002
Similarity between query and chunk 1: 0.7341793775558472
Similarity between query and chunk 2: 0.7578483819961548
Similarity between query and chunk 3: 0.7220343947410583
Similarity between query and chunk 4: 0.7406705021858215
Similarity between query and chunk 5: 0.7535263299942017
Similarity between query and chunk 6: 0.7309044599533081
Similarity between query and chunk 7: 0.7280457615852356
Similarity between query and chunk 8: 0.7647887468338013
Similarity between query and chunk 9: 0.7749450206756592
Similarity between query and chunk 10: 0.7227254509925842
Similarity between query and chunk 11: 0.7075538635253906
Similarity between query and chunk 12: 0.7429360747337341
Similarity between query and chunk 13: 0.7335089445114136
Similarity between query and chunk 14: 0.7326527833938599
Similarity between query and chunk 15: 0.7721964716911316
S

In [23]:
import numpy as np

def get_top_five_similarities_fast(query, embedding_matrix, model):
    # 1. Get the query embedding as a 1D numpy array
    query_vector = np.array(list(model.embed([query]))[0])
    
    # 2. Compute similarity for ALL chunks at once using vector dot product
    # Assumes embedding_matrix has shape (num_chunks, embedding_dim)
    # and vectors are normalized. If not normalized, divide by norms.
    scores = np.dot(embedding_matrix, query_vector)
    
    # 3. Use argsort to get indices of the highest scores
    # np.argsort returns indices from smallest to largest; [::-1] reverses it
    top_indices = np.argsort(scores)[::-1][:5]
    
    # 4. Return as a list of (index, score) tuples to match your original format
    return [(idx, float(scores[idx])) for idx in top_indices]

def similarity_comparison_top_result(query):
    query_embedding=list(model.embed([query]))[0]
    
    def get_top_similarity(embeddings):
        max_sim=-1
        top_index=-1
        for i, emb in enumerate(embeddings):
            sim=get_similarity(query_embedding, emb)
            if sim > max_sim:
                max_sim=sim
                top_index=i
        return top_index, max_sim
    
    top_fixed_index, top_fixed_sim=get_top_similarity(embeddings)
    top_sliding_index, top_sliding_sim=get_top_similarity(sliding_embeddings)
    top_header_index, top_header_sim=get_top_similarity(header_aware_embeddings)
    
    return {
        "fixed": {"index": top_fixed_index, "similarity": top_fixed_sim},
        "sliding": {"index": top_sliding_index, "similarity": top_sliding_sim},
        "header": {"index": top_header_index, "similarity": top_header_sim}
    }

def similarity_comparison_top_result_display(query):
    result = similarity_comparison_top_result(query)
    print(f"\nQuery: {query}\n")
    print(f"Top similarity with fixed size chunks: Chunk {result['fixed']['index']} (Similarity: {result['fixed']['similarity']})")
    print(f"Top similarity with sliding window chunks: Chunk {result['sliding']['index']} (Similarity: {result['sliding']['similarity']})")
    print(f"Top similarity with header aware chunks: Chunk {result['header']['index']} (Similarity: {result['header']['similarity']})")
    

def retrieve_top_chunks(query, embedding_matrix, model, processed_docs, k=5):
    chunk_indices = get_top_five_similarities_fast(query, embedding_matrix, model)
    top_chunks = []
    for idx, score in chunk_indices:
        if idx < len(processed_docs):  # Ensure the index is within bounds
            top_chunks.append({
                "chunk_index": idx,
                "similarity_score": score,
                 "section": processed_docs[idx]["metadata"]["section"] if idx < len(processed_docs) and "metadata" in processed_docs[idx] else "N/A",
                "chunk_content": processed_docs[idx]["content"]  # Assuming each chunk is a list of dicts
            })
    return top_chunks

In [24]:
similarity_comparison_top_result(query1)
similarity_comparison_top_result(query2)
similarity_comparison_top_result(query3)

{'fixed': {'index': 3, 'similarity': np.float32(0.8767275)},
 'sliding': {'index': 37, 'similarity': np.float32(0.8783426)},
 'header': {'index': 43, 'similarity': np.float32(0.8902777)}}

In [25]:
queries=["What is SGD?",
"What is mini-batch SGD?",
"What is a GPU?",
"What is an NPU?",
"What are embeddings?",
"What is vector space?",
"What is cosine similarity?",
"What is precision?",
"What is recall?",
"What is F1 score?"]

In [26]:

for q in queries:
    similarity_comparison_top_result_display(q)


Query: What is SGD?

Top similarity with fixed size chunks: Chunk 22 (Similarity: 0.8493565320968628)
Top similarity with sliding window chunks: Chunk 30 (Similarity: 0.8651751279830933)
Top similarity with header aware chunks: Chunk 35 (Similarity: 0.8750566840171814)

Query: What is mini-batch SGD?

Top similarity with fixed size chunks: Chunk 23 (Similarity: 0.9087215065956116)
Top similarity with sliding window chunks: Chunk 31 (Similarity: 0.9087215065956116)
Top similarity with header aware chunks: Chunk 36 (Similarity: 0.8828941583633423)

Query: What is a GPU?

Top similarity with fixed size chunks: Chunk 25 (Similarity: 0.8014718294143677)
Top similarity with sliding window chunks: Chunk 33 (Similarity: 0.8280395269393921)
Top similarity with header aware chunks: Chunk 39 (Similarity: 0.8382422924041748)

Query: What is an NPU?

Top similarity with fixed size chunks: Chunk 25 (Similarity: 0.8188723921775818)
Top similarity with sliding window chunks: Chunk 34 (Similarity: 0.7

In [27]:
# 1. Semantic Ambiguity Tests
semantic_ambiguity_queries = [
    "What are precision, recall, and F1 score, and why are they insufficient for evaluating a modern generative pipeline?",
    "Explain the concept of embeddings. How does a static embedding model differ from how embeddings are handled right before a Transformer's attention layer?",
    "What is the difference between a position bias in an LLM judge and positional encoding?"
]

# 2. Deep Context & Hierarchy Tests
deep_context_queries = [
    "Why is exact k-NN search impossible at a massive scale, and how do Voronoi cells in an Inverted File Index solve this?",
    "Explain the difference between factual hallucinations and faithfulness hallucinations.",
    "What happens to the gradient when the learning rate is too large versus too small?"
]

# 3. Code & Math Extraction Tests
code_math_queries = [
    "Show me the Python implementation for scaled dot-product attention and explain what the d_k variable represents.",
    "I need to build an IVF-PQ index. Provide the FAISS Python code and explain how the compression ratio is achieved.",
    "Provide the mathematical formula for Cosine Similarity and show the Python code to compute it."
]

# 4. Advanced Cross-Document Synthesis Tests
cross_document_synthesis_queries = [
    "Trace the evolution of search: How does the synonym problem in BM25 relate to the curse of dimensionality in high-dimensional vector spaces?",
    "If I am using an HNSW index, why might applying a pre-filtering metadata constraint completely destroy my search process?",
    "How do parallel execution units in GPUs specifically accelerate the scaled dot-product calculations inside a multi-head attention mechanism?"
]


In [28]:
all_experimental_queries= (
    semantic_ambiguity_queries + 
    deep_context_queries + 
    code_math_queries + 
    cross_document_synthesis_queries
)

## TESTING BRO

In [29]:
model=TextEmbedding()
print(processed_docs)
print(processed_docs[0]["content"])
embeddings=model_embedding(processed_docs)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\AI_PROJECTS\ice_pytorch\monarch\Monarch-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Laksh\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[{'content': "# Advanced Vector Search and Approximate Nearest Neighbors (ANN)\n\n## 1. The Geometry of High-Dimensional Space\n\nIn retrieval-augmented generation (RAG) pipelines, text is embedded into high-dimensional vector spaces. Modern embedding models, such as OpenAI's `text-embedding-3-large` or open-source alternatives like `BGE-m3`, frequently output vectors with 1024, 1536, or even 3072 dimensions.\n\nUnderstanding how data behaves in these spaces is critical. Human intuition is built for three-dimensional space, bu", 'metadata': {'file_name': 'advanced_vector_search.md', 'chunking_strategy': 'fixed', 'chunk_index': 0, 'start_char': 0, 'section': 'Advanced Vector Search and Approximate Nearest Neighbors (ANN)', 'words': 67, 'lines': 7}}, {'content': 't geometry behaves very differently when dealing with thousands of dimensions. \n\n### 1.1 The Curse of Dimensionality\n\nThe "Curse of Dimensionality" refers to various phenomena that arise when analyzing and organizing data in

In [31]:
# print(f"semantic_ambiguity_queries: {semantic_ambiguity_queries}")
# for i in semantic_ambiguity_queries:
#     similarity_comparison_top_result(i)
# print(f"deep_context_queries: {deep_context_queries}")
# for i in deep_context_queries:
#     similarity_comparison_top_result(i)
# print(f"code_math_queries: {code_math_queries}")
# for i in code_math_queries:
#     similarity_comparison_top_result(i)
# print(f"cross_document_synthesis_queries: {cross_document_synthesis_queries}")
# for i in cross_document_synthesis_queries:
#     similarity_comparison_top_result(i)

In [32]:
def qualitative_reranker(query, retrieved_chunks):
    boosted_chunks = []
    query_words = set(query.lower().split())
    
    for chunk in retrieved_chunks:
        score = chunk["similarity_score"]
        section_path = chunk["metadata"]["section"].lower()
        
        # Qualitative Rule: If terms from the hierarchy path are explicitly 
        # typed by the user, boost the relevance score by 15%
        path_words = set(section_path.replace(" > ", " ").split())
        matches = query_words.intersection(path_words)
        
        if matches:
            score += 0.15 * len(matches) # Add a qualitative weight bonus
            
        chunk["reranked_score"] = score
        boosted_chunks.append(chunk)
        
    # Sort by the new score
    return sorted(boosted_chunks, key=lambda x: x["reranked_score"], reverse=True)



In [33]:
a=similarity_comparison_top_result("What is the difference between a position bias in an LLM judge and positional encoding?")
print(f"Top chunk before reranking: {a}")

ValueError: shapes (384,) and (768,) not aligned: 384 (dim 0) != 768 (dim 0)

In [34]:
retrieve_top_chunks("what is similarity?", embeddings,model,processed_docs, k=5)

ValueError: shapes (51,768) and (384,) not aligned: 768 (dim 1) != 384 (dim 0)

In [35]:
retrieve_top_chunks("what is deep learning?", embeddings, model, processed_docs, k=5)

ValueError: shapes (51,768) and (384,) not aligned: 768 (dim 1) != 384 (dim 0)

In [ ]:
retrieve_top_chunks("what is recall?", embeddings, model,processed_docs, k=5)

In [ ]:
print(len(processed_docs))
print(len(embeddings))

In [ ]:
print("Retrieving top chunks for 'what is GPU?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_top_chunks("what is GPU?", header_aware_embeddings, model, header_aware_chunks, k=5))

In [ ]:
print("Retrieving top chunks for 'why do mini-batches work well on GPUs?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_top_chunks("why do mini-batches work well on GPUs?", header_aware_embeddings, model, header_aware_chunks, k=5))

In [ ]:
from collections import Counter

contents = [chunk["content"] for chunk in header_aware_chunks]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in header aware: {len(duplicates)}")

contents = [chunk["content"] for chunk in sliding_window_chunks]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in sliding window: {len(duplicates)}")

contents = [chunk["content"] for chunk in processed_docs]
duplicates = [item for item, count in Counter(contents).items() if count > 1]
print(f"Duplicate chunks in fixed: {len(duplicates)}")

print(len(header_aware_embeddings))
print(len(sliding_window_embeddings))
print(len(processed_docs))


### query rewrite

In [ ]:
from ollama import Client

client = Client(host="http://127.0.0.1:11434")
def rewrite_query(query):
    system_prompt = (
            "You are an expert search engine query rewriter. Your job is to take a short, "
            "ambiguous user query and expand it into a dense, descriptive paragraph. "
            "Include technical synonyms, underlying concepts, and relevant terms. "
            "Do not answer the query. Only return the expanded search terms."
        )
    response = client.generate(
            model='qwen3:4b',
            prompt=f"System: {system_prompt}\nUser Query: {query}\nExpanded Query:"
        )    
    return response['response'].strip()

def retrieve_with_rewriter(query, embedding_matrix, model, chunks, k=5):
    expanded_query = rewrite_query(query)
    print(f"Expanded Query: {expanded_query}")
    return retrieve_top_chunks(expanded_query, embedding_matrix, model, chunks, k)

In [ ]:
print("Retrieving top chunks for 'why do mini-batches work well on GPUs?':\n\n")
print("using fixed size chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", embeddings, model,processed_docs, k=5))
sliding_window_embeddings=model_embedding(sliding_window_chunks)
header_aware_embeddings=model_embedding(header_aware_chunks)
print("\n\nusing sliding window chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", sliding_window_embeddings, model, sliding_window_chunks, k=5))
print("\n\nusing header aware chunking embeddings:")
print(retrieve_with_rewriter("why do mini-batches work well on GPUs?", header_aware_embeddings, model, header_aware_chunks, k=5))

In [ ]:
import os
import ollama

# Force python to ignore proxies for local loops
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# Or initialize an explicit local client bypassing environment configs
from ollama import Client
client = Client(host="http://127.0.0.1:11434")

# Test connection via tags endpoint
try:
    print("Available Local Models:", client.list())
except Exception as e:
    print("Still failed:", e)

In [ ]:
RAG_prompt="""You are an expert retrieval optimizer.

Rewrite the user's query into a dense semantic search query suitable for vector search.

Requirements:

1. Retain the user's original intent.
2. Add relevant terminology, technical keywords, synonyms, and closely related concepts.
3. Expand short or ambiguous phrases into their likely meaning.
4. Include terminology that may appear in educational, technical, or documentation-style texts.
5. Avoid conversational language.
6. Do not answer the query.
7. Output only the rewritten query.

user query: {query}
"""
query="what is rag?"
prompt=RAG_prompt.format(query=query)
prompt